# Create rdf

In [1]:
from rdflib import Graph, URIRef, BNode, RDF, RDFS
from rdflib.namespace import OWL
from rdflib.collection import Collection
from itertools import combinations
from collections import defaultdict
import pandas as pd


In [2]:
base_iri = 'http://example.org/'

hn = Graph()

defined_classes = set()
property_definitions = defaultdict(lambda: {'domains': set(), 'ranges': set()})
property_map = dict()
tbox = pd.read_csv('metaedges.tsv', sep='\t')
for i,row in tbox.iterrows():
    prop_line = row['metaedge']

    if '-' in prop_line:
        prop_name = row['metaedge'].replace(' ','').split('-')[1]
        domain_str, prop_str, range_str = [item.replace(' ','') for item in prop_line.split(' - ')]

    else :
        prop_name= row['metaedge'].replace(' ','').split('>')[1]
        domain_str, prop_str, range_str = [item.replace(' ','') for item in prop_line.split(' > ')]

    property_map[row['abbreviation']] = prop_name
    property_definitions[prop_str]['domains'].add(domain_str)
    property_definitions[prop_str]['ranges'].add(range_str) # Also collect ranges for completeness

    # domain_uri = URIRef(domain_str.replace(' ',''))
    # prop_uri = URIRef(prop_str.replace(' ',''))
    # range_uri = URIRef(range_str.replace(' ',''))
    # hn.add((prop_uri, RDF.type, RDF.Property))
    # hn.add((prop_uri, RDFS.domain, domain_uri))
    # hn.add((prop_uri, RDFS.range, range_uri))

    if domain_str not in defined_classes:
        hn.add((URIRef(base_iri + domain_str.replace(' ','')), RDF.type, RDFS.Class))
        defined_classes.add(domain_str)
    if range_str not in defined_classes:
        hn.add((URIRef(base_iri + range_str.replace(' ','')), RDF.type, RDFS.Class))
        defined_classes.add(range_str)

for class_name in defined_classes:
    hn.add((URIRef(base_iri + class_name), RDF.type, RDFS.Class))

for prop_str, defs in property_definitions.items():
    prop_uri = URIRef(base_iri + prop_str.replace(' ',''))
    hn.add((prop_uri, RDF.type, RDF.Property))

    domain_uris = [URIRef(base_iri +d.replace(' ','')) for d in defs['domains']]
    if len(domain_uris) == 1:
        # Simple case: only one domain
        hn.add((prop_uri, RDFS.domain, domain_uris[0]))
    else:
        # Complex case: create a union class for the domain
        union_bnode = BNode() # An anonymous class for the union
        hn.add((union_bnode, RDF.type, RDFS.Class))
        # Use rdflib's Collection to create the RDF list for owl:unionOf
        c = Collection(hn, BNode(), domain_uris)
        hn.add((union_bnode, OWL.unionOf, c.uri))
        hn.add((prop_uri, RDFS.domain, union_bnode))

    range_uris = [URIRef(base_iri + r.replace(' ','')) for r in defs['ranges']]
    if len(range_uris) == 1:
        hn.add((prop_uri, RDFS.range, range_uris[0]))
    else:
        union_bnode = BNode()
        hn.add((union_bnode, RDF.type, RDFS.Class))
        c = Collection(hn, BNode(), range_uris)
        hn.add((union_bnode, OWL.unionOf, c.uri))
        hn.add((prop_uri, RDFS.range, union_bnode))
class_uris = [URIRef(base_iri + c.replace(' ','')) for c in defined_classes]
for class1, class2 in combinations(class_uris, 2):
    hn.add((class1, OWL.disjointWith, class2))

print(len(hn))
print(hn.serialize(format='turtle'))


152
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

_:N5038eabddc3f43b3a5961d43a45dcedd a rdfs:Class ;
    owl:unionOf ( <http://example.org/Disease> <http://example.org/Compound> ) .

_:N5baec73c79ca455eb123b3a29673f34f a rdfs:Class ;
    owl:unionOf ( <http://example.org/Disease> <http://example.org/Compound> ) .

_:N6306f3d5cd094b94975bada951d849ed a rdfs:Class ;
    owl:unionOf ( <http://example.org/CellularComponent> <http://example.org/BiologicalProcess> <http://example.org/MolecularFunction> <http://example.org/Pathway> ) .

_:Nc3fc11d091084dc9ab7ac44e8b47d122 a rdfs:Class ;
    owl:unionOf ( <http://example.org/Disease> <http://example.org/Anatomy> <http://example.org/Compound> ) .

_:Nf07277d15ba24c22b06efff2dbb210c5 a rdfs:Class ;
    owl:unionOf ( <http://example.org/Disease> <http://example.org/Anatomy> <http://example.org/Compound> ) .

<http://example.org/

In [3]:
# add individuals and types
with open('entity_mapping.nt','r') as f:

    for line in f.readlines():
        names = line.replace('<','').replace('>','')
        s,p,o,_ = names.split(' ')
        hn.add((URIRef(base_iri + s), RDF.type ,URIRef(base_iri + o)))

print(len(hn))

45310


In [4]:
# add individual relations from train

with open('hetio_train.tsv','r') as f:
    cnt = 0
    for line in f.readlines():
        cnt += 1
        s,p,o = line.strip().split('\t')
        hn.add((URIRef(base_iri + s), URIRef(base_iri + property_map.get(p)) ,URIRef(base_iri + o)))

hn.serialize('hetio_train_graph.nt', format='nt')

/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N48aa6a3ec3e74f85a5cc712721efcf37 (<class 'rdflib.graph.Graph'>)>

In [5]:
with open('hetio_validation.tsv','r') as f:
    cnt = 0
    for line in f.readlines():
        cnt += 1
        s,p,o = line.strip().split('\t')
        hn.add((URIRef(base_iri +s), URIRef(base_iri +property_map.get(p)) ,URIRef(base_iri +o)))
with open('hetio_test.tsv', 'r') as f:
    cnt = 0
    for line in f.readlines():
        cnt += 1
        s, p, o = line.strip().split('\t')
        hn.add((URIRef(base_iri +s), URIRef(base_iri +property_map.get(p)), URIRef(base_iri +o)))

hn.serialize('hetio_full_graph.nt', format='nt')

/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N48aa6a3ec3e74f85a5cc712721efcf37 (<class 'rdflib.graph.Graph'>)>

In [6]:
# make nicer datasets
for df_name in ['hetio_train.tsv', 'hetio_validation.tsv', 'hetio_test.tsv']:
    pre_df = pd.read_csv(df_name, sep='\t', header=None, names=['s', 'p','o'])
    pre_df['nice_p'] = pre_df.p.apply(lambda x: base_iri +property_map.get(x))
    pre_df['nice_s'] = pre_df.s.apply(lambda x: base_iri +x)
    pre_df['nice_o'] = pre_df.o.apply(lambda x: base_iri +x)
    pre_df[['nice_s','nice_p','nice_o']].to_csv(df_name.replace('.','_nice.'), sep='\t', index=False, header=False)

In [32]:
for f_name in ['hetio_train_graph.nt', 'hetio_full_graph.nt']:
    with open(f_name, 'r') as in_f:
        with open(f_name.replace('.','_noIRI.'), 'w') as out_f:
            for line in in_f:
                out_f.write(line.replace('<','').replace('>','') + '\n')

In [25]:
from owlready2 import *

In [26]:
world = World()
onto = get_ontology("hetio_full_graph.nt").load()

In [27]:
with onto:
    sync_reasoner()

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/owlready2/hermit:/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/ct/yy2gltkx6n56mgt7r9gf4qsm0000gn/T/tmphe7a1vuk
* Owlready2 * HermiT took 14.925103187561035 seconds
* Owlready * Equivalenting: owl.bottomObjectProperty T.participates
* Owlready * Equivalenting: owl.bottomObjectProperty T.resembles
* Owlready * Equivalenting: T.participates owl.bottomObjectProperty
* Owlready * Equivalenting: T.participates T.resembles
* Owlready * Equivalenting: T.resembles owl.bottomObjectProperty
* Owlready * Equivalenting: T.resembles T.participates
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


# Time to materialize

In [7]:
tnice = pd.read_csv('hetio_train_nice.tsv', sep='\t', names = ['s', 'p','o'])

In [8]:
tnice[tnice.p.str.contains('-')]

,s,p,o


In [9]:
property_map

{'AdG': 'downregulates',
 'AeG': 'expresses',
 'AuG': 'upregulates',
 'CbG': 'binds',
 'CcSE': 'causes',
 'CdG': 'downregulates',
 'CpD': 'palliates',
 'CrC': 'resembles',
 'CtD': 'treats',
 'CuG': 'upregulates',
 'DaG': 'associates',
 'DdG': 'downregulates',
 'DlA': 'localizes',
 'DpS': 'presents',
 'DrD': 'resembles',
 'DuG': 'upregulates',
 'GcG': 'covaries',
 'GiG': 'interacts',
 'GpBP': 'participates',
 'GpCC': 'participates',
 'GpMF': 'participates',
 'GpPW': 'participates',
 'Gr>G': 'regulates',
 'PCiC': 'includes'}